# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tabassumrafiq/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Research question

Can a small set of observed content and search-performance signals be used to prioritize content items for human review when the observed trend indicates decline?

### Decision supported

The analysis supports a review-prioritization decision: which content items should receive attention first.

The model output is treated as a decision-support ranking signal, not as proof that an item will decline or that a specific content change will improve future performance.

In [ ]:
# Research question and decision supported

research_question = (
    "Can observed content and search-performance signals "
    "prioritize content items for human review when the observed "
    "trend indicates decline?"
)

decision_supported = (
    "Prioritize content items for human review using a model-based "
    "ranking signal."
)

print("Research question:")
print(research_question)

print("\nDecision supported:")
print(decision_supported)

Research question:
Can observed content and search-performance signals prioritize content items for human review when the observed trend indicates decline?

Decision supported:
Prioritize content items for human review using a model-based ranking signal.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*



This work uses the FlyRank ML Internship starter dataset:

`data/raw/content_refresh_anonymized.csv`

The dataset contains 30,000 anonymized content-item rows and 44 columns.

The modeling unit is one pseudonymized content item.

The analysis uses trailing-90-day content and search-performance metrics available in the dataset.

The target is derived from the observed trend direction:

- `down` → declining label
- other observed directions → non-declining label

Trend and future-outcome fields are excluded from the model features because they would directly expose the outcome being predicted or introduce leakage.

Client identifiers are used only for grouped validation and are not used as predictive features.

In [ ]:
import pandas as pd
import numpy as np

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "tabassumrafiq/flyrank-ml-internship/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

print("Dataset shape:", df.shape)

# Target definition
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget rate:")
print(round(df["is_declining_label"].mean(), 4))

Dataset shape: (30000, 44)

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target rate:
0.5421


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Methodology

#### Features

The modeling lane uses five observed features:

- `avg_position`
- `ctr`
- `engagement_rate`
- `scroll_rate`
- `word_count`

These features were selected from the available content and search-performance signals after excluding outcome and future-oriented fields.

#### Label

The binary target is:

`is_declining_label = 1` when `trend_direction == "down"`.

This label represents an observed trend direction rather than a guaranteed future outcome.

#### Baseline

The baseline prioritizes content using observed search visibility and average search position.

It is used as a simple decision rule to establish a comparison point before relying on the learned model.

#### Model

The modeling lane uses a Random Forest classifier with median imputation.

The model's predicted probability for the declining class is used as the ranking score.

#### Validation

Validation uses a client-grouped split so that clients in the test set do not overlap with clients in the training set.

The grouped result is treated as the more conservative estimate of generalization to unseen clients.

#### Leakage checks

Trend, future, outcome, label, and refresh-decision fields are excluded from the predictive feature set.

Client identifiers are used for grouping only and are not predictive features.

No client names, private queries, or other private identifiers are used in the paper.

In [ ]:
features = [
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "word_count"
]

target = "is_declining_label"

X = df[features].copy()
y = df[target].copy()

print("Features:")
for feature in features:
    print("-", feature)

print("\nTarget:", target)
print("Feature matrix shape:", X.shape)

Features:
- avg_position
- ctr
- engagement_rate
- scroll_rate
- word_count

Target: is_declining_label
Feature matrix shape: (30000, 5)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=200,
            max_depth=6,
            random_state=42,
            n_jobs=-1
        )
    )
])

model.fit(X, y)

df["decline_score"] = model.predict_proba(X)[:, 1]

print("Model trained successfully.")
print("Decline scores generated:", len(df))

Model trained successfully.
Decline scores generated: 30000


## 4. Results (vs Baseline)

### Baseline

The ML-07 baseline is a transparent content-review prioritization rule.

It uses two observed signals:

- `impressions_90d`
- `avg_position`

Each signal is converted to a percentile rank and the two ranks are combined
with equal weight. The top 10% of the resulting score distribution is marked
for human review.

After removing rows with missing or invalid baseline inputs, the baseline
contained 28,795 valid content items.

The baseline is a ranking rule rather than a binary classifier. Therefore,
Accuracy, Precision, Recall, and F1 were not calculated for the baseline.

### Learned Model

The ML-08 Random Forest model uses multiple observed features to produce a
model-based decline probability that can be used as a ranking signal.

The measured results were:

| Metric | Random Split | Client-Grouped Split |
|---|---:|---:|
| Accuracy | 0.6352 | 0.5398 |
| Precision | 0.6121 | 0.5283 |
| Recall | 0.8924 | 0.9270 |
| F1 | 0.7261 | 0.6730 |

The client-grouped split had no client overlap between training and test data.
It is therefore treated as the more conservative estimate of generalization
to unseen clients.

### Interpretation

The baseline provides a simple and transparent reference for prioritization,
while the Random Forest provides a learned ranking signal using multiple
observed features.

The grouped validation result shows a measurable directional signal, but the
lower grouped performance indicates that the model should be treated as
decision-support rather than production-grade prediction.

Neither the baseline nor the model proves that an individual content item will
decline or that a particular refresh action will improve future performance.

In [1]:
# Results vs baseline summary

baseline_valid_rows = 28795
baseline_features = [
    "impressions_90d",
    "avg_position"
]

model_results = {
    "Random split": {
        "Accuracy": 0.6352,
        "Precision": 0.6121,
        "Recall": 0.8924,
        "F1": 0.7261
    },
    "Client-grouped split": {
        "Accuracy": 0.5398,
        "Precision": 0.5283,
        "Recall": 0.9270,
        "F1": 0.6730
    }
}

print("BASELINE")
print("Valid rows:", baseline_valid_rows)
print("Features:", baseline_features)
print("Review rule: Top 10% of baseline score")

print("\nMODEL — CLIENT-GROUPED VALIDATION")

for metric, value in model_results["Client-grouped split"].items():
    print(f"{metric}: {value}")

BASELINE
Valid rows: 28795
Features: ['impressions_90d', 'avg_position']
Review rule: Top 10% of baseline score

MODEL — CLIENT-GROUPED VALIDATION
Accuracy: 0.5398
Precision: 0.5283
Recall: 0.927
F1: 0.673


## 5. Limitations & Honest Framing

This analysis has several important limitations.

### 1. Observational data

The analysis uses observed historical content and search-performance signals.
The relationships identified by the model should therefore be interpreted as
directional associations rather than causal effects.

### 2. Decline label

The target label is derived from the observed `trend_direction` field.

A declining label describes an observed trend in the available data. It does
not guarantee that an item will continue to decline in the future.

### 3. Generalization

The client-grouped validation is a more conservative estimate of
generalization to unseen clients, but it does not establish performance in
every future dataset or operating environment.

### 4. Baseline limitations

The ML-07 baseline uses only 90-day impressions and average search position.
It is intentionally simple and does not represent all factors that may affect
content performance.

### 5. Model limitations

The Random Forest combines several observed features into a learned signal,
but the model does not establish why an item is declining or which specific
content intervention will work.

### 6. Human review

The output is intended for decision-support and prioritization.

A human reviewer should consider additional context before taking action,
including content relevance, quality, recent changes, search intent, and
strategic importance.

### What this work does NOT claim

This work does not claim to:

- predict Google's ranking algorithm;
- prove that a specific factor causes content decline;
- guarantee future traffic or ranking changes;
- prove that refreshing content will improve performance;
- automatically determine which content should be rewritten, deleted, or
  published.

The appropriate interpretation is that the workflow provides a measurable,
directional signal that can help prioritize content for human review.

In [2]:
# Limitations checklist

limitations = {
    "Observational data": True,
    "Observed trend label": True,
    "Client-grouped validation": True,
    "Baseline limitations documented": True,
    "Human review required": True,
    "Causal claims avoided": True,
    "Google ranking claims avoided": True,
}

print("Limitations and honest-framing checks:")

for item, passed in limitations.items():
    print(f"{item}: {'PASS' if passed else 'CHECK'}")

Limitations and honest-framing checks:
Observational data: PASS
Observed trend label: PASS
Client-grouped validation: PASS
Baseline limitations documented: PASS
Human review required: PASS
Causal claims avoided: PASS
Google ranking claims avoided: PASS


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

. Ranked Recommendations

The model output is converted into a ranked review queue. The purpose of the
queue is to help content teams decide which items deserve attention first.

### Recommendation 1 — Prioritize high-scoring items for review

Items with higher model-based decline scores should be considered earlier in
the human-review queue.

A high score is a prioritization signal, not confirmation that an item needs a
refresh.

### Recommendation 2 — Review the underlying signals before acting

Reviewers should inspect the observed search and content-performance signals
that contributed to the ranking.

The model score should be considered together with content relevance, quality,
search intent, recent changes, and strategic importance.

### Recommendation 3 — Use reason codes to support review

The action playbook should provide a clear reason code for each prioritized
item so that reviewers can understand why the item entered the queue.

Reason codes are intended to make the workflow more interpretable and easier
to audit.

### Recommendation 4 — Treat refresh decisions as human decisions

A reviewer should decide whether the appropriate action is to improve,
rewrite, merge, monitor, or take no action.

The model should not automatically rewrite, delete, publish, or refresh content.

### Recommendation 5 — Monitor the workflow over time

The ranking system should be reviewed when the underlying data distribution,
feature behavior, label definition, or validation performance changes
meaningfully.

Model retraining should be considered when monitoring shows that the existing
model no longer provides a useful decision-support signal.

### Action Priority

| Priority | Action | Reason |
|---|---|---|
| 1 | Review high-scoring items | Stronger model-based prioritization signal |
| 2 | Inspect contributing signals | Understand the observed evidence |
| 3 | Apply reason codes | Improve transparency and auditability |
| 4 | Select an appropriate content action | Requires human judgment |
| 5 | Monitor performance and data drift | Detect when the workflow becomes stale |

These recommendations are directional and intended for decision-support. They
do not establish that a particular intervention will improve future search
performance.

In [4]:
import pandas as pd

# Ranked recommendation summary

recommendations = [
    {
        "priority": 1,
        "action": "Review high-scoring items",
        "reason": "Stronger model-based prioritization signal"
    },
    {
        "priority": 2,
        "action": "Inspect contributing signals",
        "reason": "Understand the observed evidence"
    },
    {
        "priority": 3,
        "action": "Apply reason codes",
        "reason": "Improve transparency and auditability"
    },
    {
        "priority": 4,
        "action": "Select an appropriate content action",
        "reason": "Requires human judgment"
    },
    {
        "priority": 5,
        "action": "Monitor performance and data drift",
        "reason": "Detect when the workflow becomes stale"
    }
]

recommendations_df = pd.DataFrame(recommendations)

display(recommendations_df)

,priority,action,reason
0,1,Review high-scoring items,Stronger model-based prioritization signal
1,2,Inspect contributing signals,Understand the observed evidence
2,3,Apply reason codes,Improve transparency and auditability
3,4,Select an appropriate content action,Requires human judgment
4,5,Monitor performance and data drift,Detect when the workflow becomes stale


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

 Artifacts the Paper Embeds

The paper uses a small set of artifacts to make the methodology and results
auditable.

### Artifact 1 — Decline Score Distribution

The decline-score distribution shows how the model-based scores are distributed
across the evaluated content items.

This figure is used as a descriptive artifact rather than evidence of causal
impact.

### Artifact 2 — Validation Results

The validation results table compares the measured model performance under a
random split and a client-grouped split.

The client-grouped result is emphasized because it provides a more conservative
test of generalization to unseen clients.

### Artifact 3 — Ranked Action Queue

The action playbook produces a ranked queue with model-based scores, reason
codes, and review actions.

The queue is intended for human review and decision support.

In [5]:
# Validation results table for the paper

results_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1"],
    "Random Split": [0.6352, 0.6121, 0.8924, 0.7261],
    "Client-Grouped Split": [0.5398, 0.5283, 0.9270, 0.6730]
})

display(results_df)

,Metric,Random Split,Client-Grouped Split
0,Accuracy,0.6352,0.5398
1,Precision,0.6121,0.5283
2,Recall,0.8924,0.9270
3,F1,0.7261,0.6730


In [6]:
# Baseline summary

baseline_summary = pd.DataFrame({
    "Baseline component": [
        "90-day impressions",
        "Average search position",
        "Scoring method",
        "Review threshold"
    ],
    "Description": [
        "Percentile rank",
        "Percentile rank",
        "Equal-weight sum of percentile ranks",
        "Top 10%"
    ]
})

display(baseline_summary)

,Baseline component,Description
0,90-day impressions,Percentile rank
1,Average search position,Percentile rank
2,Scoring method,Equal-weight sum of percentile ranks
3,Review threshold,Top 10%


### Figure

The following previously generated ML-10 artifact can be reused in the
deployed paper:

`work/figures/decline_score_distribution.png`

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [ ]:
checks = {
    "question_defined": len(research_question) > 0,
    "data_loaded": len(df) == 30000,
    "features_present": all(
        feature in df.columns for feature in features
    ),
    "target_present": target in df.columns,
    "scores_generated": "decline_score" in df.columns,
    "figure_created": figure_path.exists()
}

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

assert all(checks.values())

print("\nCAPSTONE SELF-CHECK: PASS")

question_defined: PASS
data_loaded: PASS
features_present: PASS
target_present: PASS
scores_generated: PASS
figure_created: PASS

CAPSTONE SELF-CHECK: PASS


# ML-12 — Tell the Story

## 5-Minute Demo Outline

### 1. Question — 45 seconds

Can observed content and search-performance signals prioritize content items
for human review when the observed trend indicates decline?

The decision supported by this work is review prioritization. The model is used
as decision support and does not make the final editorial decision.

### 2. Method — 1 minute

The analysis uses the anonymized FlyRank ML Internship dataset.

The workflow:

- ML-06 audited observed search and content signals.
- ML-07 created a transparent baseline action score.
- ML-08 trained a machine-learning model.
- ML-09 validated the model and audited the research claims.
- ML-10 converted the model output into a ranked content action playbook.
- ML-11 deployed the results as a public research paper.

Client-grouped validation was used to evaluate generalization to unseen clients.

### 3. One Chart — 45 seconds

The decline score distribution shows how the model's ranking scores are
distributed across the evaluated content items.

Higher scores can be used to move items toward earlier human review.

The score is a prioritization signal, not proof of future decline or proof that
a specific intervention will succeed.

### 4. One Honest Result — 1 minute

Client-grouped validation produced:

- Accuracy: 0.5398
- Precision: 0.5283
- Recall: 0.9270
- F1: 0.6730

The grouped result is more conservative than the random-split result because
test clients were not present during training.

### 5. One Recommendation — 1 minute

Prioritize high-scoring content items for human review.

Before acting, a reviewer should consider content relevance, quality, recent
changes, strategic importance, and whether an intervention is appropriate.

The model should not automatically rewrite, delete, publish, or refresh content.

---

## ML-06 + ML-07 — Signal Audit and Baseline

### ML-06 — Signal Audit

The signal audit evaluated key content and search signals against observed
decline labels. Average position and 90-day impressions were directionally
confirmed, while word count showed only a small difference and was classified
as MIXED. The results support using search visibility and ranking position as
decision-support signals, not as proof of content decline.

### ML-07 — Baseline Action Score

A transparent baseline score was created using 90-day impressions and average
search position. Both signals were converted to percentile ranks and combined
to prioritize the top 10% of content for review. The resulting ranked queue and
top-20 review provide a simple, explainable baseline for comparison with the
later ML model.

### Connection to the Capstone

ML-06 validated the directional usefulness of key search signals, while ML-07
converted the strongest observed signals into a transparent baseline action
score. This baseline provides an interpretable benchmark for evaluating whether
the subsequent machine-learning model improves content-review prioritization.

---

## Shareable Cut — Social Post

I built a machine-learning workflow for content-review prioritization using an
anonymized FlyRank ML Internship dataset.

The workflow uses observed search and engagement signals to rank content items
for human review instead of treating a model score as an automatic decision.

I compared a learned model with a transparent baseline and used client-grouped
validation to test generalization to unseen clients.

The grouped validation produced an F1 score of 0.6730, with 0.9270 recall and
0.5283 precision.

The main lesson was that useful ML output needs more than a prediction: it
needs honest validation, clear limitations, and a practical human-review
workflow.

---

## Shareable Cut — Employer Summary

I built a content-review prioritization workflow using an anonymized FlyRank
ML Internship dataset with 30,000 content-item rows and 44 columns.

The workflow combines signal auditing, a transparent baseline, machine-learning
classification, client-grouped validation, and a ranked action playbook to
turn observed content-performance signals into a human-review queue.

The grouped validation produced an F1 score of 0.6730, showing a measurable
directional signal while also highlighting limitations in generalization to
unseen clients.